# 01 - Bronze : CSV DVF brut vers Delta (RustFS)

Copie **fidèle** du fichier DVF dans le data lake : toutes les colonnes en texte, aucune ligne retirée, aucune transformation.
Le nombre de lignes obtenu ici est le point de départ de la silver.

In [1]:
import sys
sys.path.append("/home/jovyan/work")

from outils.source import creer_session, creer_bucket, RAW, LAKE

ANNEE = 2025
CSV = f"{RAW}/full-{ANNEE}.csv"
BRONZE = f"{LAKE}/bronze/dvf_{ANNEE}"

## Session Spark et bucket

In [2]:
spark = creer_session("bronze")
creer_bucket()

## Lecture du CSV (tout en texte)

In [3]:
df = spark.read.option("header", True).option("inferSchema", False).csv(CSV)
print(len(df.columns), "colonnes")

## Écriture en Delta

In [4]:
df.write.format("delta").mode("overwrite").save(BRONZE)

## Contrôle : relecture depuis le lac

In [5]:
n_csv = df.count()
n_bronze = spark.read.format("delta").load(BRONZE).count()
print(f"CSV : {n_csv:,} lignes | bronze : {n_bronze:,} lignes".replace(",", " "))
assert n_csv == n_bronze, "La bronze n'a pas le même nombre de lignes que le CSV."

In [6]:
spark.stop()